# Tantra-LLM / NP-DNA — GPU Training on Google Colab

Resumes the 131K-vocab GPU checkpoint on a Colab T4 and continues training.

## Before you run
1. Install the repo somewhere Colab can reach it (GitHub clone in cell 3, **or** copy the whole `Tantra-LLM` folder into `MyDrive/Tantra-LLM` and cell 3 copies from Drive).
2. Upload your **latest GPU checkpoint** so it lands at `model/latest` on Colab.
3. Provide the training data in `Download/` (combined jsonl folders or a `train_pack/*.jsonl`).

The training code already supports Drive backup via the `NPDNA_BACKUP_DIR` env var, so every `latest` save is mirrored back to Drive.

In [ ]:
#@title 1. Runtime config
DRIVE_CPY = "/content/drive/MyDrive/Tantra-LLM"   #@param {type:"string"}
SOURCE_FROM_GITHUB = True                        #@param {type:"boolean"}
GIT_URL = "https://github.com/atulyaai/Tantra-LLM.git"  #@param {type:"string"}
PROJECT_DIR = "/content/Tantra-LLM"
CKPT_NAME = "latest"                            #@param ["latest","best"]
BACKUP_DIR = "/content/drive/MyDrive/Tantra_Checkpoints"


In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, shutil

if SOURCE_FROM_GITHUB and not os.path.exists(PROJECT_DIR):
    os.system(f"git clone --depth 1 {GIT_URL} {PROJECT_DIR}")
elif not SOURCE_FROM_GITHUB:
    shutil.copytree(DRIVE_CPY, PROJECT_DIR, dirs_exist_ok=True)

%cd {PROJECT_DIR}
print("PROJECT_DIR =", PROJECT_DIR)

In [ ]:
#@title Verify GPU
import torch, subprocess, sys
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM GB:", torch.cuda.get_device_properties(0).total_memory/1e9)
    subprocess.run([sys.executable, "-c", "import numpy; print('cuda:'+torch.version.cuda)"])
else:
    print("Runtime -> Change runtime type -> T4 GPU, then re-run.")

In [ ]:
import pathlib, IPython.display
ckpt_src = pathlib.Path(BACKUP_DIR) / CKPT_NAME

if ckpt_src.exists():
    print("FOUND checkpoint on Drive, copying into training slot...")
    dst = pathlib.Path(PROJECT_DIR) / "model" / CKPT_NAME
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(ckpt_src, dst)
    print("Copied to", dst)
else:
    print("No checkpoint at", ckpt_src)
    print("\nUpload the checkpoint FOLDER so it lands at:")
    print("  MyDrive/Tantra_Checkpoints/latest/{model.pt, tokenizer.json, metadata.json, training_state.pt, cortex/}")
    IPython.display.display(IPython.display.FileUpload())


In [ ]:
from pathlib import Path
p = Path(PROJECT_DIR)/"model"/CKPT_NAME
for f in p.rglob("*"):
    if f.is_file():
        print(f"{f.relative_to(p)}  {f.stat().st_size/1e6:.1f} MB")

In [ ]:
import subprocess, sys
reqs = Path(PROJECT_DIR)/"requirements.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(reqs)], check=True)

In [ ]:
#@title 8. Training hyper-parameters
TARGET_STEPS = 100000             #@param {type:"integer"}
BATCH_SIZE = 8                   #@param {type:"integer"}
SEQ_LEN = 256                    #@param {type:"integer"}
GRAD_ACCUM = 4                   #@param {type:"integer"}
LR = 3e-3                        #@param {type:"number"}
USE_AMP = True                   #@param {type:"boolean"}
USE_COMPILE = False              #@param {type:"boolean"}
GRAD_CHECKPOINT = False          #@param {type:"boolean"}
AUTO_GROW = False                #@param {type:"boolean"}


In [ ]:
import os
os.environ["NPDNA_BACKUP_DIR"] = BACKUP_DIR
os.makedirs(BACKUP_DIR, exist_ok=True)

flags = [
    f"--target-steps {TARGET_STEPS}",
    f"--batch-size {BATCH_SIZE}",
    f"--seq-len {SEQ_LEN}",
    f"--grad-accum-steps {GRAD_ACCUM}",
    f"--lr {LR}",
    "--device cuda",
    f"--resume-from {CKPT_NAME}",
]
if USE_AMP: flags.append("--amp")
if USE_COMPILE: flags.append("--compile")
if GRAD_CHECKPOINT: flags.append("--grad-checkpoint")
if AUTO_GROW: flags.append("--auto-grow")

cmd = "python -m npdna.train " + " ".join(flags)
print(cmd)

In [ ]:
import subprocess, sys, os
os.environ["NPDNA_BACKUP_DIR"] = BACKUP_DIR
proc = subprocess.run([sys.executable, "-m", "npdna.train"] + cmd.replace("python -m npdna.train ", "").split(),
                      cwd=PROJECT_DIR)
print("Exit code:", proc.returncode)

In [ ]:
#@title Resume later: re-run cell 9 (uses --resume-from latest).
# The latest save is already mirrored to Drive by NPDNA_BACKUP_DIR,
# so re-mounting a fresh runtime and re-running from cell 8 continues cleanly.
!echo "Done. Checkpoints are in MyDrive/Tantra_Checkpoints/latest"